# Messy Crime Dataset — Data Cleaning & Exploratory Analysis

## Project Overview

This project focuses on cleaning, validating, and analyzing a synthetic crime dataset containing incident, officer, suspect, victim, location, case status, and financial information.

The dataset was intentionally created with various data quality issues, including missing values, inconsistent categorical labels, malformed numeric values, mixed date formats, invalid numerical values, duplicate records, and inconsistent text formatting.

The main objective of this project is to transform the raw dataset into a cleaner and more reliable analytical dataset while maintaining traceability and avoiding unsupported assumptions.

## Objectives

* Identify and assess data quality issues in the raw dataset.
* Clean inconsistent categorical and textual values.
* Handle invalid numerical and datetime values.
* Remove duplicate records and validate unique incident IDs.
* Standardize data types where appropriate.
* Validate the cleaned dataset using explicit data-quality rules.
* Perform exploratory analysis to identify patterns and distributions in crime incidents.
* Document the cleaning decisions and limitations of the dataset.

## Dataset

| Item                                  | Description                   |
| ------------------------------------- | ----------------------------- |
| Original records                      | 5,250                         |
| Final records                         | 5,050                         |
| Columns                               | 33                            |
| Duplicate rows after cleaning         | 0                             |
| Duplicate incident IDs after cleaning | 0                             |
| Dataset type                          | Synthetic crime incident data |

## Analytical Workflow

**Inspect → Clean → Validate → Analyze → Visualize → Document**

The cleaning process follows a conservative approach:

> Values are corrected when the intended value can be reasonably determined from the available data. When a value cannot be reliably corrected, it is treated as missing rather than replaced with an unsupported assumption.

## Tools

* Python
* Pandas
* NumPy
* Matplotlib
* Jupyter Notebook


# Import Libraries & Dataset

The analysis is performed using Python with Pandas and NumPy for data manipulation and Matplotlib for visualization.

The raw dataset is loaded first and preserved as the original reference before the cleaning process begins.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("crime_incidents_messy.csv")

In [ ]:
df_clean = df.copy()

# Initial Data Inspection

Before cleaning, the dataset was inspected to understand its structure, data types, missing values, duplicate records, unique categories, and potential data-quality issues.

The inspection focused on identifying problems that could affect the reliability of subsequent analysis.

### Initial Checks

The following aspects were examined:

* Dataset dimensions and column structure
* Data types
* Missing values
* Duplicate records
* Duplicate incident IDs
* Unique categorical values
* Numerical ranges and potential outliers
* Inconsistent text formatting
* Mixed datetime formats
* Malformed numeric values


In [ ]:
print("Rows   :", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
missing_summary = (
    df.isna()
      .sum()
      .to_frame("missing")
      .assign(
          missing_pct=lambda x: (x["missing"] / len(df) * 100).round(2)
      )
      .sort_values("missing", ascending=False)
)

missing_summary

In [ ]:
print("Duplicate rows :", df.duplicated().sum())
print("Duplicate IDs  :", df["incident_id"].duplicated().sum())

In [ ]:
categorical_columns = [
    "crime_type",
    "district",
    "severity",
    "case_status",
    "resolution",
    "weapon_used",
    "suspect_gender",
    "suspect_race",
    "victim_gender"
]

for col in categorical_columns:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))

# Data Cleaning

The cleaning process was performed systematically based on the issues identified during the initial inspection.

The objective was not to force every missing or inconsistent value into a valid category. Instead, each transformation was evaluated based on the available evidence and the expected domain of the variable.

The main cleaning steps included:

1. Removing duplicate records.
2. Standardizing datetime values.
3. Correcting invalid numerical values where the intended value could be determined.
4. Converting unrecoverable numerical values to missing.
5. Correcting recoverable geographic coordinate errors.
6. Standardizing categorical labels and correcting clear spelling variations.
7. Standardizing selected text fields.
8. Converting columns to appropriate data types.

### Cleaning Principle

> Correct when the intended value can be reasonably determined; otherwise, preserve the uncertainty as a missing value.


### 1. Duplicate Records

Duplicate records were identified using both complete-row duplication and the `incident_id` field.

Complete duplicate rows were removed because they represent repeated records of the same observation. The `incident_id` field was subsequently checked to ensure that each incident remained uniquely represented.


In [ ]:
before = len(df_clean)

df_clean = df_clean.drop_duplicates()

after = len(df_clean)

print("Rows before :", before)
print("Rows after  :", after)
print("Removed     :", before - after)

In [ ]:
print("Duplicate rows:", df_clean.duplicated().sum())
print("Duplicate IDs :", df_clean["incident_id"].duplicated().sum())

### 2. Datetime Cleaning

The `incident_datetime` column contained multiple date formats, including datetime strings, slash-separated dates, and date-only values.

The values were parsed according to their observed formats and converted into a consistent datetime representation. Values that could not be reliably parsed were retained as missing.

A final check was also performed to ensure that no incident date occurred in the future.


In [ ]:
df_clean['incident_datetime'].head(25)

In [ ]:
# see how many dates use the format with slashes (/) and how many use dashes (-)
df_clean['incident_datetime'].str.contains('/').sum(), df_clean['incident_datetime'].str.contains('-').sum()

In [ ]:
df_clean[
    df_clean['incident_datetime'].str.contains('/')
]['incident_datetime'].head(25)

In [ ]:
condition = (
    ~df_clean['incident_datetime'].str.match(
        r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}',
        na=False
    ) & df_clean['incident_datetime'].notna()
)
print(df_clean[condition]['incident_datetime'].head(25))

In [ ]:
iso_condition = df_clean['incident_datetime'].str.match(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}',na=False)

df_clean.loc[iso_condition, 'incident_datetime'].head(25)

In [ ]:
slash_condition = df_clean['incident_datetime'].str.contains('/',na=False)

df_clean.loc[slash_condition,['incident_datetime']].head(25)

In [ ]:
other_condition = (
    ~iso_condition
    &
    ~slash_condition
    &
    df_clean['incident_datetime'].notna()
)

print(other_condition.sum())

In [ ]:
# 1. Change column tracking type to loose objects
df_clean['incident_datetime'] = df_clean['incident_datetime'].astype(object)

In [ ]:
df_clean.loc[iso_condition, 'incident_datetime'] = pd.to_datetime(
    df_clean.loc[iso_condition, 'incident_datetime'],
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
)

print(df_clean.loc[iso_condition, 'incident_datetime'].head())

In [ ]:
df_clean.loc[slash_condition, 'incident_datetime'] = pd.to_datetime(
    df_clean.loc[slash_condition, 'incident_datetime'],
    format='%m/%d/%Y %H:%M',
    errors='coerce'
)

print(df_clean.loc[slash_condition,'incident_datetime'].head())

In [ ]:
df_clean.loc[other_condition, 'incident_datetime'] = pd.to_datetime(
    df_clean.loc[other_condition, 'incident_datetime'],
    format='%d-%m-%Y',
    errors='coerce'
)

print(df_clean.loc[other_condition, 'incident_datetime'].head(25))

In [ ]:
df_clean['incident_datetime'] = pd.to_datetime(
    df_clean['incident_datetime']
)

### 3. Numerical Data Cleaning

#### 3.1 Badge Number

The `badge_number` column was converted to Pandas' nullable integer type so that integer values could be preserved while allowing missing values.

In [ ]:
df_clean["badge_number"] = df_clean["badge_number"].astype("Int64")

#### 3.2 Number of Arrests

The `num_arrests` column contained negative values even though the variable represents a count of arrests.

After examining the affected values, the negative values were corrected to their corresponding positive counts. The column was then converted to Pandas' nullable integer type.

In [ ]:
negative_filter_num = df_clean[df_clean["num_arrests"] < 0]

df_clean.loc[
    negative_filter_num.index,
    "num_arrests"
] = abs(
    df_clean.loc[
        negative_filter_num.index,
        "num_arrests"
    ]
)

In [ ]:
df_clean["num_arrests"] = df_clean["num_arrests"].astype("Int64")

#### 3.3 Suspect Age

The `suspect_age` column contained implausible values, including negative ages and values outside the defined valid range.

Based on the observed data and the analytical scope of the dataset, ages from 15 to 75 were treated as valid. Values outside this range were converted to missing rather than forcibly corrected.

In [ ]:
negative_age = df_clean[df_clean['suspect_age'] < 0]

df_clean.loc[negative_age.index,'suspect_age'] = abs(
    df_clean.loc[negative_age.index,'suspect_age']
)

In [ ]:
high_age = df_clean['suspect_age'] > 100

df_clean.loc[high_age, 'suspect_age'] = pd.NA

#### 3.4 Victim Age

The `victim_age` column contained negative and implausibly high values.

After reviewing the lower age values, ages from 10 to 90 were considered valid for this analysis. Values outside this range were converted to missing.

In [ ]:
negative_victim_age = df_clean[
    df_clean['victim_age'] < 0
    ]

df_clean.loc[negative_victim_age.index, 'victim_age'] = abs(
    df_clean.loc[negative_victim_age.index, 'victim_age']
)

high_victim_age = df_clean['victim_age'] > 100
df_clean.loc[high_victim_age, 'victim_age'] = pd.NA

#### 3.5 Property Loss

The `property_loss_usd` column contained malformed numeric strings, including values with duplicated decimal segments.

The malformed values were cleaned based on the observed formatting pattern before the column was converted to a numeric data type. Values that could not be reliably interpreted were retained as missing.

In [ ]:
df_clean["property_loss_usd"] = pd.to_numeric(
    df_clean["property_loss_usd"],
    errors="coerce"
)

In [ ]:
invalid_property_loss = df[
    df["property_loss_usd"].notna()
    & df_clean["property_loss_usd"].isna()
].copy()

In [ ]:
invalid_property_loss['property_loss_usd'] = (
    invalid_property_loss['property_loss_usd']
    .str.replace(r'\.0$','', regex=True)
)

In [ ]:
invalid_property_loss["property_loss_usd"] = pd.to_numeric(
    invalid_property_loss["property_loss_usd"],
    errors="coerce"
)

In [ ]:
df_clean.loc[invalid_property_loss.index, "property_loss_usd"] = (
    invalid_property_loss["property_loss_usd"]
)

In [ ]:
negative_filter = df_clean[df_clean['property_loss_usd'] < 0]

df_clean.loc[negative_filter.index, 'property_loss_usd'] = abs(
    df_clean.loc[negative_filter.index, 'property_loss_usd']
)

### 4. Geographic Coordinate Cleaning

The `latitude` and `longitude` columns were evaluated against their valid geographic ranges.

Some records contained values that appeared to have their latitude and longitude fields swapped. When the intended coordinates could be reasonably determined from the valid ranges, the values were corrected by swapping the two fields.

Records with coordinates that could not be reliably recovered were converted to missing values.

The final dataset contains no out-of-range latitude or longitude values.


In [ ]:
invalid_latitude = df[
    (df["latitude"] < -90) |
    (df["latitude"] > 90)
]
invalid_latitude[["incident_id", "latitude", "longitude"]]

In [ ]:
invalid_longitude = df[
    (df["longitude"] < -180) |
    (df["longitude"] > 180)
]
invalid_longitude[["incident_id", "latitude", "longitude"]]

In [ ]:
invalid_latitude = df_clean[
    (df["latitude"] < -90) |
    (df["latitude"] > 90)
]
print(
    invalid_latitude[
        ['latitude', 'longitude']
    ]
)

In [ ]:
# Correct recoverable latitude/longitude swaps
swapped_coordinates = df_clean[
    (
        (df_clean['latitude'] < -90) |
        (df_clean['latitude'] > 90)
    ) &
    (
        df_clean['longitude'].between(-90, 90)
    ) &
    (
        df_clean['latitude'].between(-180, 180)
    )
]
print(swapped_coordinates.shape[0])
print(
    swapped_coordinates[
        ['latitude', 'longitude']
    ].head(25)
)

In [ ]:
swapped_lat = swapped_coordinates['longitude']
swapped_long = swapped_coordinates['latitude']

In [ ]:
other_invalid_latitude = df_clean[
    (
        (df_clean['latitude'] < -90) |
        (df_clean['latitude'] > 90)
    ) &
    ~(
        (df_clean['longitude'].between(-90, 90))
        &
        (df_clean['latitude'].between(-180,180))
    )
    
]
print(other_invalid_latitude.shape)
print(
    other_invalid_latitude[
        ['latitude', 'longitude']
    ].head(25)
)

In [ ]:
invalid_latitude = (
    (df_clean['latitude'] < -90) |
    (df_clean['latitude'] > 90)
)

swapped_coordinates = (
    invalid_latitude
    &
    df_clean['longitude'].between(-90, 90)
    &
    df_clean['latitude'].between(-180, 180)
)

print("Invalid latitude :", invalid_latitude.sum())
print("Swappable        :", swapped_coordinates.sum())
print(
    "Unrecoverable     :",
    (invalid_latitude & ~swapped_coordinates).sum()
)

In [ ]:
# 55 Rows Swappable latitude & longitude 
df_clean.loc[swapped_coordinates, ['latitude','longitude']] = (
    df_clean.loc[swapped_coordinates, ['longitude','latitude']]
    .to_numpy()
)

# Set unrecoverable coordinates to missing
uncoverable_latitude = invalid_latitude & ~swapped_coordinates
df_clean.loc[uncoverable_latitude, 'latitude'] = pd.NA

In [ ]:
# Validation
print(
    (
        (df_clean['latitude'] < -90) |
        (df_clean['latitude'] > 90)
    ).sum()
)
print(df_clean['latitude'].isna().sum())
print(
    (
        (df_clean['longitude'] < -180) |
        (df_clean['longitude'] > 180)
    ).sum()
)
print(df_clean['longitude'].isna().sum())

### 5. Categorical Data Cleaning

#### 5.1 Crime Type

The `crime_type` column contained inconsistent capitalization, spelling variations, abbreviations, and alternative naming conventions.

The values were first normalized for formatting and then reviewed for clear spelling and naming variations.

Only categories that could be reasonably identified as the same crime type were merged. Semantically different categories were intentionally kept separate to avoid over-standardization.

In [ ]:
crime_type_normalized = (
    df_clean['crime_type']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.lower()
)
print(crime_type_normalized.value_counts())
print(crime_type_normalized.nunique())

In [ ]:
print(
    sorted(
        crime_type_normalized.dropna().unique()
    )
)

In [ ]:
# Mapping typo/abbreviation

crime_type_mapping = {
    # Spelling corrections
    'arsen': 'arson',
    'asslt': 'assault',
    'b&e': 'breaking & entering',
    'burglry': 'burglary',
    'cybercrime': 'cyber crime',
    'homocide': 'homicide',
    'kidnaping': 'kidnapping',
    'robbry': 'robbery',
    'roberry': 'robbery',
    'sexual assualt': 'sexual assault',
    'tresspassing': 'trespassing',
    'vandlism': 'vandalism',

    # Abbreviations
    'dom. violence': 'domestic violence',
    'domestc violence': 'domestic violence',
    'dv': 'domestic violence',

    'd.u.i.': 'dui',
    'duii': 'dui',
    'dwi': 'dui',
    'drunk driving': 'dui',

    'sa': 'sexual assault',

    # Drug-related naming
    'drug offense': 'drug offence',
    'drugs': 'drug offence',
    'narcotics': 'drug offence'
}

df_clean['crime_type'] = (
    crime_type_normalized
    .replace(crime_type_mapping)
)

In [ ]:
# Validasi Mapping crime_type
print(df_clean['crime_type'].nunique())
print(df_clean['crime_type'].value_counts())
print(df_clean['crime_type'].isna().sum())

#### 5.2 District

The `district` column contained inconsistent formatting and abbreviated district names.

Clear abbreviations such as `cen`, `eas`, `nor`, `sou`, and `wes` were standardized to their corresponding district names.

Categories that could not be confidently identified as duplicates were retained separately.

In [ ]:
district_normalized = (
    df_clean['district']
    .str.strip()
    .str.replace(r'\s+',' ',regex=True)
    .str.lower()
)

print(district_normalized.nunique())
print(
    district_normalized
    .value_counts()
    .sort_index()
)
print(
    sorted(
        district_normalized.unique()
    )
)

In [ ]:
# Mapping district
district_mapping = {
    'cen': 'central',
    'eas': 'east',
    'nor': 'north',
    'sou': 'south',
    'wes': 'west'
}
df_clean['district'] = district_normalized.replace(district_mapping)

print(df_clean['district'].nunique())
print(df_clean['district'].value_counts())
print(df_clean['district'].isna().sum())

#### 5.3 Severity

The `severity` column was standardized to consistent category labels while preserving missing values.

In [ ]:
df_clean["severity"].value_counts(dropna=False)

In [ ]:
severity_map = {
    "1": "Low",
    "2": "Medium",
    "3": "High",
    "4": "Critical",
    "low": "Low",
    "med": "Medium",
    "medium": "Medium",
    "high": "High",
    "crit": "Critical",
    "critical": "Critical"
}

In [ ]:
df_clean["severity"] = (
    df_clean["severity"]
    .astype("string")
    .str.strip() # penghapusan spasi di kiri/kanan data "     1"
    .str.lower() # standarisasi semua huruf menjadi lowercase
    .map(severity_map)
)

In [ ]:
df_clean["severity"].value_counts(dropna=False)

In [ ]:
print("df:", len(df))
print("df_clean:", len(df_clean))
print("Duplicate df_clean:", df_clean.duplicated().sum())

#### 5.4 Case Status

The `case_status` column contained spelling inconsistencies and abbreviated forms.

Clear variants were standardized to a consistent set of case-status categories.

In [ ]:
df['case_status'].value_counts(dropna=False)

In [ ]:
df_clean['case_status_normalize'] = (
    df_clean['case_status'].astype('string').str.strip().str.lower()
)

df_clean['case_status_normalize'].value_counts(dropna=False)

In [ ]:
# mapping untuk memperbaiki typo

case_status_map = {
    'open' : 'Open',
    'closed' : 'Closed',
    'resolved' : 'Resolved',
    'pendng' : 'Pending',
    'pending' : 'Pending',
    'under investigation' : 'Under Investigation',
    'investgation' : 'Under Investigation'
}

df_clean['case_status'] = (
    df_clean['case_status_normalize'].map(case_status_map)
)

In [ ]:
df_clean = df_clean.drop(columns='case_status_normalize')

In [ ]:
df_clean["case_status"].value_counts(dropna=False)

#### 5.5 Resolution

The `resolution` column contained inconsistent labels and spelling variations.

Clear variants were standardized while missing values were retained.

In [ ]:
df_clean["resolution"].value_counts(dropna=False)

In [ ]:
df_clean['resolution_normalized'] = (
    df_clean['resolution'].astype('string').str.strip().str.lower()
)

df_clean['resolution_normalized'].value_counts(dropna=False)

In [ ]:
resolution_map = {
    'arrest made' : 'Arrest Made',
    'no arrest' : 'No Arrest',
    'warning' : 'Warning Issued',
    'warning issued' : 'Warning Issued',
    'dismissed' : 'Dismissed',
    'case dismissed' : 'Dismissed',
    'arres made' : 'Arrest Made',
}

df_clean['resolution'] = (
    df_clean['resolution_normalized'].map(resolution_map)
)

In [ ]:
df_clean  = df_clean.drop(columns='resolution_normalized')

In [ ]:
df_clean['resolution'].value_counts(dropna=False)

#### 5.6 Gender and Race

Gender and race fields contained capitalization inconsistencies and abbreviated representations.

Gender values such as `M`, `m`, `F`, and `f` were standardized to their corresponding full labels. Race categories were standardized for capitalization consistency.

Unknown and missing values were preserved rather than inferred.

In [ ]:
df_clean['suspect_gender'].value_counts(dropna=False)

In [ ]:
df_clean['suspect_gender'] = (
    df_clean['suspect_gender']
    .str.strip()
    .str.title()
)
df_clean['suspect_gender'].value_counts(dropna=False)

In [ ]:
gender_mapping = {
    'F': 'Female',
    'M': 'Male'
}

df_clean['suspect_gender'] = df_clean['suspect_gender'].replace(gender_mapping)

In [ ]:
df_clean['suspect_gender'].value_counts(dropna=False)

In [ ]:
df_clean['victim_gender'].value_counts(dropna=False)

In [ ]:
df_clean['victim_gender'] = (
    df_clean['victim_gender']
    .str.strip()
    .str.title()
)

In [ ]:
victim_gender_mapping = {
    'F': 'Female',
    'M': 'Male'
}

df_clean['victim_gender'] = df_clean['victim_gender'].replace(
    victim_gender_mapping
)

In [ ]:
df_clean['victim_gender'].value_counts(dropna=False)

In [ ]:
df_clean['suspect_race'].value_counts(dropna=False)

In [ ]:
df_clean['suspect_race'] = (
    df_clean['suspect_race']
    .str.strip()
    .str.title()
)
df_clean['suspect_race'].value_counts(dropna=False)

#### 5.7 Weapon Used

The `weapon_used` column contained capitalization inconsistencies across several categories.

Clear capitalization variants were standardized while categories that could not be confidently considered equivalent were retained separately.

In [ ]:
print("Missing:", df_clean['weapon_used'].isna().sum())
print("Unique :", df_clean['weapon_used'].nunique())

df_clean['weapon_used'].value_counts(dropna=False).head(25)

In [ ]:
df_clean['weapon_used'].value_counts(dropna=False)

In [ ]:
weapon_mapping = {
    'KNIFE': 'Knife',
    'blunt object': 'Blunt Object',
    'firearm': 'Firearm'
}

df_clean['weapon_used'] = df_clean['weapon_used'].replace(weapon_mapping)

In [ ]:
df_clean['weapon_used'].value_counts(dropna=False)

In [ ]:
hands_check = df_clean[
    df_clean['weapon_used'].isin(['hands', 'Hands/Feet'])
][['weapon_used', 'crime_type']]

pd.crosstab(
    hands_check['weapon_used'],
    hands_check['crime_type']
)

In [ ]:
print("Unique:", df_clean['weapon_used'].nunique())
print("Missing:", df_clean['weapon_used'].isna().sum())

df_clean['weapon_used'].value_counts(dropna=False)

### 6. Text Standardization

Selected text fields were standardized to improve consistency while preserving the underlying information.

The main operations included removing unnecessary whitespace, standardizing capitalization, and normalizing phone-number formatting.

No values were inferred solely from other records when the intended value could not be established reliably.


#### 6.1 Officers Names

In [ ]:
first_name_normalized = (
    df_clean['officer_first_name']
    .str.strip()
    .str.lower()
)
print("Before :", df_clean['officer_first_name'].nunique())
print("After :", first_name_normalized.nunique())

In [ ]:
df_clean['officer_first_name'] = (
    df_clean['officer_first_name']
    .str.strip()
    .str.title()
)
print("Unique:", df_clean['officer_first_name'].nunique())
print(df_clean['officer_first_name'].value_counts().head(10))

In [ ]:
last_name_normalized = (
    df_clean['officer_last_name']
    .str.strip()
    .str.lower()
)

print("Before :", df_clean['officer_last_name'].nunique())
print("After :", last_name_normalized.nunique())

#### 6.2 victim_phone

In [ ]:
print("Missing:", df_clean['victim_phone'].isna().sum())
print("Unique :", df_clean['victim_phone'].nunique())

df_clean['victim_phone'].value_counts(dropna=False).head(20)

In [ ]:
phone_digits_only = (
    df_clean['victim_phone']
    .dropna()
    .str.fullmatch(r'\d+')
)

phone_digits_only.value_counts()

In [ ]:
phone_length = (
    df_clean['victim_phone']
    .dropna()
    .str.replace(r'\D', '', regex=True)
    .str.len()
)

phone_length.value_counts().sort_index()

In [ ]:
df_clean['victim_phone'] = (
    df_clean['victim_phone']
    .str.replace(r'\D', '', regex=True)
)

In [ ]:
print("Missing:", df_clean['victim_phone'].isna().sum())

phone_length = (
    df_clean['victim_phone']
    .dropna()
    .str.len()
)

print(phone_length.value_counts())

### 7. Final Data Type Standardization

After the individual cleaning operations were completed, the final data types were reviewed to ensure that each column uses an appropriate representation for analysis.

Nullable Pandas data types were used where appropriate to preserve missing values while maintaining numeric or boolean semantics.


In [ ]:
dtype_summary = pd.DataFrame({
    "column": df_clean.columns,
    "dtype": df_clean.dtypes.astype(str).values,
    "missing": df_clean.isna().sum().values
})

dtype_summary

# Data Validation

After the cleaning process, the dataset was validated against predefined data-quality rules.

The validation checks focus on structural integrity, valid numerical ranges, geographic coordinates, datetime consistency, and duplicate records.

A record is considered invalid only when it contains a non-missing value that violates the defined validation rule. Missing values are not treated as invalid because some fields could not be reliably recovered during cleaning.

The validation results are summarized below.

In [ ]:
# Structural validation
duplicate_rows = df_clean.duplicated().sum()
duplicate_ids = df_clean["incident_id"].duplicated().sum()

# Geographic validation
invalid_lat = (
    df_clean["latitude"].notna()
    & ~df_clean["latitude"].between(-90, 90)
).sum()

invalid_lon = (
    df_clean["longitude"].notna()
    & ~df_clean["longitude"].between(-180, 180)
).sum()

# Age validation
invalid_suspect_age = (
    df_clean["suspect_age"].notna()
    & ~df_clean["suspect_age"].between(15, 75)
).sum()

invalid_victim_age = (
    df_clean["victim_age"].notna()
    & ~df_clean["victim_age"].between(10, 90)
).sum()

# Arrest validation
invalid_num_arrests = (
    df_clean["num_arrests"].notna()
    & ~df_clean["num_arrests"].between(0, 5)
).sum()

# Financial validation
invalid_property_loss = (
    df_clean["property_loss_usd"].notna()
    & (df_clean["property_loss_usd"] < 0)
).sum()

# Datetime validation
future_dates = (
    df_clean["incident_datetime"].notna()
    & (df_clean["incident_datetime"] > pd.Timestamp.now())
).sum()

In [ ]:
validation_results = pd.DataFrame({
    "Check": [
        "Duplicate rows",
        "Duplicate incident IDs",
        "Invalid latitude",
        "Invalid longitude",
        "Invalid suspect age",
        "Invalid victim age",
        "Invalid number of arrests",
        "Invalid property loss",
        "Future incident dates"
    ],
    "Rule": [
        "Each complete record should appear once",
        "Each incident_id should be unique",
        "Latitude must be between -90 and 90",
        "Longitude must be between -180 and 180",
        "Age must be between 15 and 75",
        "Age must be between 10 and 90",
        "Arrests must be between 0 and 5",
        "Property loss must be non-negative",
        "Incident date must not be in the future"
    ],
    "Invalid records": [
        duplicate_rows,
        duplicate_ids,
        invalid_lat,
        invalid_lon,
        invalid_suspect_age,
        invalid_victim_age,
        invalid_num_arrests,
        invalid_property_loss,
        future_dates
    ]
})

validation_results["Status"] = np.where(
    validation_results["Invalid records"] == 0,
    "PASS",
    "CHECK"
)

validation_results

In [ ]:
assert (validation_results["Invalid records"] == 0).all()

print("All validation checks passed.")

# Final Deliverable

The final output of this project is a validated analytical dataset, accompanied by documented cleaning decisions, data-quality checks, and exploratory visualizations.


In [ ]:
summary = {
    "Rows": len(df_clean),
    "Columns": df_clean.shape[1],
    "Duplicate Rows": df_clean.duplicated().sum(),
    "Duplicate IDs": df_clean["incident_id"].duplicated().sum(),
    "Crime Types": df_clean["crime_type"].nunique(),
    "Districts": df_clean["district"].nunique(),
    "Cities": df_clean["city"].nunique()
}

summary

# Cleaning Summary

The following summary documents the main transformations applied during the cleaning process.

The objective was to improve consistency and analytical usability without introducing unsupported assumptions. Missing values were retained when the original information could not be reliably recovered.

| Area                   | Approach                                                                            |
| ---------------------- | ----------------------------------------------------------------------------------- |
| Duplicate records      | Removed complete duplicate rows and validated unique incident IDs                   |
| Datetime               | Standardized mixed datetime formats and converted the field to datetime             |
| Numerical values       | Corrected identifiable errors and converted unrecoverable values to missing         |
| Geographic coordinates | Recovered identifiable swapped coordinates and removed unrecoverable invalid values |
| Categorical values     | Standardized clear formatting, spelling, and abbreviation variations                |
| Text fields            | Removed unnecessary whitespace and standardized selected formats                    |
| Data types             | Converted fields to appropriate numeric, boolean, and datetime representations      |
| Missing values         | Preserved when no reliable basis for imputation was available                       |

## Final Dataset

After cleaning, the dataset contains **5,050 records and 33 columns**.

The final dataset contains no duplicate rows, no duplicate incident IDs, and no values violating the defined numerical, geographic, and datetime validation rules.


In [ ]:
cleaning_summary = pd.DataFrame({
    "Column": [
        "incident_datetime",
        "reported_online",
        "badge_number",
        "num_arrests",
        "suspect_age",
        "victim_age",
        "latitude / longitude",
        "property_loss_usd",
        "severity",
        "case_status",
        "resolution",
        "crime_type",
        "district",
        "officer_first_name",
        "victim_phone",
        "weapon_used",
        "suspect_gender",
        "suspect_race",
        "victim_gender"
    ],
    "Action": [
        "Standardized mixed date formats and converted to datetime",
        "Converted to nullable boolean",
        "Converted to nullable integer",
        "Corrected negative values and converted to nullable integer",
        "Converted values outside 15–75 to missing",
        "Converted values outside 10–90 to missing",
        "Corrected recoverable swapped coordinates; unrecoverable values set to missing",
        "Cleaned malformed numeric strings and converted to float",
        "Standardized category labels",
        "Standardized category labels and corrected clear typos",
        "Standardized category labels and corrected clear typos",
        "Standardized spelling, abbreviations, and naming variations",
        "Standardized abbreviations and formatting",
        "Removed unnecessary whitespace and standardized capitalization",
        "Standardized populated values to 10-digit format",
        "Standardized clear capitalization variants",
        "Standardized F/M and capitalization variants",
        "Standardized capitalization",
        "Standardized F/M and capitalization variants"
    ]
})

cleaning_summary["Final State"] = [
    "datetime64[us]; 329 missing",
    "boolean; 486 missing",
    "Int64; 301 missing",
    "Int64; 324 missing",
    "15–75; 1,208 missing",
    "10–90; 717 missing",
    "Invalid values resolved; 372 latitude and 278 longitude missing",
    "float64; 414 missing",
    "4 categories; 342 missing",
    "5 categories; 696 missing",
    "4 categories; 912 missing",
    "35 standardized categories",
    "11 standardized categories",
    "30 unique first names",
    "4,031 populated values; all 10 digits",
    "10 categories",
    "4 categories; 1,352 missing",
    "6 categories; 1,525 missing",
    "4 categories; 963 missing"
]

cleaning_summary

### 1. Data Quality Limitations

The cleaning process improves the consistency and analytical usability of the dataset, but some source-data limitations remain.

Several identifier-to-attribute relationships were found to be inconsistent. For example, some identifiers do not consistently map to a single set of descriptive attributes across records. These relationships were not forcibly reconstructed because the available data does not provide sufficient evidence to determine which value should be considered correct.

Some semantically related categories were also intentionally kept separate when merging them would require an unsupported assumption.

Missing values were retained when reliable imputation was not possible.

Therefore, the final dataset should be considered **cleaned and validated according to the defined analytical rules**, rather than a perfect reconstruction of the original relational data.


In [ ]:
final_summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Duplicate rows",
        "Duplicate incident IDs",
        "Crime types",
        "Districts",
        "Cities",
        "Missing values"
    ],
    "Value": [
        len(df_clean),
        df_clean.shape[1],
        df_clean.duplicated().sum(),
        df_clean["incident_id"].duplicated().sum(),
        df_clean["crime_type"].nunique(),
        df_clean["district"].nunique(),
        df_clean["city"].nunique(),
        df_clean.isna().sum().sum()
    ]
})

final_summary

In [ ]:
assert df_clean.shape == (5050, 33)
assert df_clean.duplicated().sum() == 0
assert df_clean["incident_id"].duplicated().sum() == 0

print("Final dataset validation completed successfully.")

# Exploratory Data Analysis

The exploratory data analysis examines the distribution, temporal patterns, geographic differences, severity, and selected numerical characteristics of crime incidents.

The analysis is based on the cleaned and validated dataset. Visualizations are designed to highlight meaningful patterns while avoiding unnecessary chart complexity.

The main analytical questions are:

* Which crime types occur most frequently?
* How does the number of incidents change over time?
* How are incidents distributed across districts?
* How are incidents distributed by severity?
* At what times do incidents occur most frequently?
* What does the age distribution of suspects and victims look like?
* How does property loss vary across crime types?
* How does the average number of arrests vary across crime types?


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter


plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 16,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10
})

### 1. Crime Distribution

This visualization shows the number of recorded incidents for each crime type.

A horizontal bar chart is used because the dataset contains multiple crime categories, making category labels easier to read.


In [ ]:
crime_counts = (
    df_clean["crime_type"]
    .value_counts()
    .sort_values()
)

fig, ax = plt.subplots(figsize=(10, 10))

bars = ax.barh(
    crime_counts.index,
    crime_counts.values
)

ax.bar_label(
    bars,
    labels=[f"{value:,}" for value in crime_counts.values],
    padding=4,
    fontsize=9
)

ax.set_title(
    "Crime Incidents by Crime Type",
    loc="left",
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Number of incidents")
ax.set_ylabel("")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

### 2. Crime Trend Over Time

This visualization examines the annual number of recorded crime incidents from 2018 to 2024.

The year is derived from `incident_datetime` for analysis purposes and is not added as a permanent column to the cleaned dataset.


In [ ]:
# 1. Setup with a slightly softer background color for a modern feel
year_counts = df_clean["incident_datetime"].dt.year.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10, 6), facecolor="#ffffff")
ax.set_facecolor("#ffffff")

primary_color = "#2c7bb6" # A professional, deep teal/blue

# 2. Main plot with styled markers
ax.plot(
    year_counts.index,
    year_counts.values,
    marker="o",
    linewidth=2.5,
    color=primary_color,
    markersize=8,
    markeredgecolor="white",
    markeredgewidth=1.5
)

# 3. Add visual weight with a subtle fill under the line
ax.fill_between(
    year_counts.index,
    year_counts.values,
    alpha=0.1,
    color=primary_color
)

# 4. Styled Annotations
for year, value in zip(year_counts.index, year_counts.values):
    ax.annotate(
        f"{value:,}",
        (year, value),
        xytext=(0, 12), # Pushed slightly higher
        textcoords="offset points",
        ha="center",
        fontsize=10,
        fontweight="bold",
        color="#333333"
    )

# 5. Title and Subtitle Storytelling
ax.set_title(
    "Crime Incidents by Year",
    loc="left",
    fontweight="bold",
    fontsize=15,
    color="#1a1a1a",
    pad=25
)
# Adding a subtitle for context
ax.text(
    0, 1.04,
    "Annual incident volume over the tracked period", 
    transform=ax.transAxes,
    fontsize=11,
    color="#666666"
)

# 6. Axis Labels
ax.set_xlabel("Year", color="#666666", fontweight="bold", labelpad=10)
# Optional: You can remove the ylabel entirely since the title and annotations explain it
ax.set_ylabel("Number of incidents", color="#666666", fontweight="bold", labelpad=10)

# 7. Clean up ticks and format Y-axis
ax.set_xticks(year_counts.index)
ax.tick_params(colors="#666666", which="both", length=0) # Hides tick lines, keeps labels

# Format Y-axis to 'K' (Thousands) to reduce visual clutter
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, p: f'{x/1000:.0f}K' if x >= 1000 else f'{x:.0f}')
)

# 8. Declutter Spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False) # Hide left spine since we have grid lines
ax.spines["bottom"].set_color("#cccccc")

# 9. Softer Grid Lines
ax.grid(axis="y", linestyle="-", alpha=0.15, color="black")
ax.set_axisbelow(True)

# 10. Add padding to the top so annotations don't get cut off
ymin, ymax = ax.get_ylim()
ax.set_ylim(ymin, ymax * 1.15) 

plt.tight_layout()
plt.show()

### 3. Crime Distribution by District

This visualization compares the number of recorded incidents across districts.

The purpose is to identify differences in incident volume between districts, rather than to infer crime rates or population-adjusted risk.


In [ ]:
district_counts = df_clean["district"].value_counts().sort_values()

# 1. Clean the categorical labels (Capitalize for a professional look)
districts = district_counts.index.str.title()
values = district_counts.values

# 2. Setup figure with a clean white background
fig, ax = plt.subplots(figsize=(10, 7), facecolor="#ffffff")
ax.set_facecolor("#ffffff")

# 3. Strategic Color Palette
# Highlight the highest value, use a neutral gray for the rest
highlight_color = "#2c7bb6" 
neutral_color = "#e0e0e0"
colors = [neutral_color if val < max(values) else highlight_color for val in values]

# 4. Plot the horizontal bars
bars = ax.barh(
    districts,
    values,
    color=colors,
    edgecolor="none",
    height=0.7 # Slightly thinner bars for elegance
)

# 5. Add bold labels directly to the bars
ax.bar_label(
    bars,
    labels=[f"{value:,}" for value in values],
    padding=8,
    fontsize=11,
    fontweight="bold",
    color="#333333"
)

# 6. Title and Subtitle Storytelling
ax.set_title(
    "Crime Incidents by District",
    loc="left",
    fontweight="bold",
    fontsize=15,
    color="#1a1a1a",
    pad=25
)
ax.text(
    0, 1.04,
    "The North district accounts for the highest volume of reported incidents",
    transform=ax.transAxes,
    fontsize=11,
    color="#666666"
)

# 7. Extreme Data-Ink Ratio Improvement
# Hide the X-axis completely since we have direct bar labels
ax.get_xaxis().set_visible(False)

# Remove all spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.spines["left"].set_visible(False)

# Remove Y-axis ticks but keep the clean text labels
ax.tick_params(axis="y", which="both", length=0, labelsize=11, colors="#333333")

# 8. Dynamic right padding so the longest bar label doesn't get cut off
ax.set_xlim(0, max(values) * 1.15) 

plt.tight_layout()
plt.show()

### 4. Severity Distribution

This visualization shows the distribution of incidents across the four standardized severity categories.

The chart provides an overview of how incidents are classified by severity within the dataset.


In [ ]:
# 1. Define the variable from your dataframe first
severity_counts = df_clean["severity"].value_counts().reindex(["Low", "Medium", "High", "Critical"])

fig, ax = plt.subplots(figsize=(9, 6), facecolor="#ffffff")
ax.set_facecolor("#ffffff")

# 2. Semantic Color Palette (Cool/Neutral to Warm/Severe)
# Low: Gray-blue, Medium: Yellow, High: Orange, Critical: Red
severity_colors = ["#90a4ae", "#ffd54f", "#ff9800", "#e53935"]

# 3. Plot the vertical bars
bars = ax.bar(
    severity_counts.index,
    severity_counts.values,
    color=severity_colors,
    width=0.6, # Sleeker bars
    edgecolor="none"
)

# 4. Add bold labels directly above the bars
ax.bar_label(
    bars,
    labels=[f"{value:,}" for value in severity_counts.values],
    padding=8,
    fontsize=11,
    fontweight="bold",
    color="#333333"
)

# 5. Title and Subtitle Storytelling
ax.set_title(
    "Crime Incidents by Severity",
    loc="left",
    fontweight="bold",
    fontsize=15,
    color="#1a1a1a",
    pad=25
)
ax.text(
    0, 1.05,
    "Distribution of incidents scaled by assigned risk level",
    transform=ax.transAxes,
    fontsize=11,
    color="#666666"
)

# 6. Extreme Data-Ink Ratio Improvement
# Hide the Y-axis completely since we have direct bar labels
ax.get_yaxis().set_visible(False)
ax.set_xlabel("") # The categories speak for themselves

# Remove spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color("#cccccc")

# 7. Clean up X-axis ticks
ax.tick_params(axis="x", which="both", length=0, labelsize=12, colors="#333333")

# 8. Dynamic top padding to prevent label cutoff
ax.set_ylim(0, max(severity_counts.values) * 1.15)

plt.tight_layout()
plt.show()

### Initial EDA Findings

The initial exploration reveals several patterns in the cleaned dataset:

* Crime incidents are distributed across 35 crime types, with DUI, drug offence, and domestic violence among the most frequently recorded categories.
* Annual incident counts remain within a relatively narrow range between 2018 and 2024, with 2024 recording the highest number of incidents in the dataset.
* Incident volume differs across districts, with North and South containing the largest numbers of recorded incidents. These differences should not be interpreted as crime rates without population or exposure data.
* Medium and Critical incidents represent substantial portions of the dataset, while High incidents are comparatively less frequent.

These observations describe patterns within the dataset and should not be interpreted as causal relationships.


### 5. Incident Timing

This visualization examines the distribution of recorded incidents across the 24 hours of the day.

The hour is derived from `incident_datetime`. Records with missing datetime values are excluded from this analysis.

The purpose is to identify periods of the day with relatively higher or lower incident volume.


In [ ]:
hour_counts = (
    df_clean["incident_datetime"]
    .dt.hour
    .value_counts()
    .sort_index()
)

# Make sure all 24 hours are represented
hour_counts = hour_counts.reindex(range(24), fill_value=0)

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(
    hour_counts.index,
    hour_counts.values,
    marker="o",
    linewidth=2
)

ax.fill_between(
    hour_counts.index,
    hour_counts.values,
    alpha=0.12
)

ax.set_title(
    "Crime Incidents by Hour of Day",
    loc="left",
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Hour of day")
ax.set_ylabel("Number of incidents")

ax.set_xticks(range(24))
ax.set_xticklabels([f"{hour:02d}:00" for hour in range(24)], rotation=45)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

### 6. Age Distribution

#### 6.1 Suspect Age Distribution

This histogram shows the distribution of recorded suspect ages after invalid values were removed during the cleaning process.

The median age is shown as a reference point to provide additional context for the distribution.


In [ ]:
suspect_age = df_clean["suspect_age"].dropna()

fig, ax = plt.subplots(figsize=(10, 6), facecolor="#ffffff")
ax.set_facecolor("#ffffff")

primary_color = "#2c7bb6"
median_color = "#e53935" # Strong red to draw the eye

bins = np.arange(15, 76, 5)

# 1. Plot the histogram with slight transparency and crisp edges
counts, edges, patches = ax.hist(
    suspect_age,
    bins=bins,
    color=primary_color,
    edgecolor="white",
    linewidth=1.2,
    alpha=0.85 
)

# 2. Add the median line
median_suspect_age = suspect_age.median()

ax.axvline(
    median_suspect_age,
    color=median_color,
    linestyle="--",
    linewidth=2.5,
    zorder=3 # Ensures the line is drawn on top of the bars
)

# 3. Direct Labeling (Replacing the Legend)
# Place the text dynamically near the top of the highest bar
max_count = counts.max()
ax.annotate(
    f"Median Age: {median_suspect_age:.0f}",
    xy=(median_suspect_age, max_count * 0.95),
    xytext=(10, 0), # Offset 10 points to the right
    textcoords="offset points",
    va="center",
    color=median_color,
    fontweight="bold",
    fontsize=11
)

# 4. Title and Subtitle Storytelling
ax.set_title(
    "Suspect Age Distribution",
    loc="left",
    fontweight="bold",
    fontsize=15,
    color="#1a1a1a",
    pad=25
)
ax.text(
    0, 1.04,
    "The majority of suspects fall below the overall median age",
    transform=ax.transAxes,
    fontsize=11,
    color="#666666"
)

# 5. Axis Formatting
ax.set_xlabel("Age", color="#666666", fontweight="bold", labelpad=10)
ax.set_ylabel("Number of suspects", color="#666666", fontweight="bold", labelpad=10)

# Align x-ticks exactly with the bin edges for perfect readability
ax.set_xticks(bins)
ax.tick_params(colors="#666666", which="both", length=0)

# 6. Declutter Spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color("#cccccc")

# 7. Grid Setup
ax.grid(axis="y", linestyle="-", alpha=0.15, color="black")
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

#### 6.2 Victim Age Distribution

This histogram shows the distribution of recorded victim ages after invalid values were removed during the cleaning process.

The median age is shown to provide a reference point for the center of the distribution.


In [ ]:
victim_age = df_clean["victim_age"].dropna()

fig, ax = plt.subplots(figsize=(10, 6), facecolor="#ffffff")
ax.set_facecolor("#ffffff")

# A distinct color (Indigo) to separate Victim data from Suspect data (Blue)
primary_color = "#3949ab" 
median_color = "#e53935"

bins = np.arange(10, 91, 5)

# 1. Plot the histogram
counts, edges, patches = ax.hist(
    victim_age,
    bins=bins,
    color=primary_color,
    edgecolor="white",
    linewidth=1.2,
    alpha=0.85 
)

# 2. Calculate and plot the median line
median_victim_age = victim_age.median()

ax.axvline(
    median_victim_age,
    color=median_color,
    linestyle="--",
    linewidth=2.5,
    zorder=3
)

# 3. Direct Labeling for the Median
max_count = counts.max()
ax.annotate(
    f"Median Age: {median_victim_age:.0f}",
    xy=(median_victim_age, max_count * 0.95),
    xytext=(10, 0),
    textcoords="offset points",
    va="center",
    color=median_color,
    fontweight="bold",
    fontsize=11
)

# 4. Contextual Titles
ax.set_title(
    "Victim Age Distribution",
    loc="left",
    fontweight="bold",
    fontsize=15,
    color="#1a1a1a",
    pad=25
)
ax.text(
    0, 1.04,
    "Distribution of victim ages across all reported incidents",
    transform=ax.transAxes,
    fontsize=11,
    color="#666666"
)

# 5. Clean Axis Labels and Ticks
ax.set_xlabel("Age", color="#666666", fontweight="bold", labelpad=10)
ax.set_ylabel("Number of victims", color="#666666", fontweight="bold", labelpad=10)

ax.set_xticks(bins)
ax.tick_params(colors="#666666", which="both", length=0)

# 6. Declutter Spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color("#cccccc")

# 7. Background Grid
ax.grid(axis="y", linestyle="-", alpha=0.15, color="black")
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

### 7. Property Loss by Crime Type

This analysis compares median reported property loss across crime types.

The median is used instead of the mean because it provides a more robust measure of the typical property loss when distributions may contain unusually high or low values.

Only crime types with at least 100 valid property-loss observations are included to provide a more comparable basis across categories.


In [ ]:
property_loss_by_crime = (
    df_clean
    .dropna(subset=["property_loss_usd"])
    .groupby("crime_type")["property_loss_usd"]
    .agg(
        median="median",
        count="count"
    )
)

property_loss_by_crime = (
    property_loss_by_crime[
        property_loss_by_crime["count"] >= 100
    ]
    .sort_values("median")
)

fig, ax = plt.subplots(figsize=(10, 8))

bars = ax.barh(
    property_loss_by_crime.index,
    property_loss_by_crime["median"]
)

ax.bar_label(
    bars,
    labels=[
        f"${value:,.0f}"
        for value in property_loss_by_crime["median"]
    ],
    padding=4,
    fontsize=9
)

ax.set_title(
    "Median Property Loss by Crime Type",
    loc="left",
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Median property loss (USD)")
ax.set_ylabel("")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

### 8. Average Arrests by Crime Type

This analysis compares the average number of arrests across crime types.

Only crime types with at least 100 valid arrest observations are included to reduce the influence of categories with very small sample sizes.

The metric represents the average number of arrests recorded per incident in the dataset.


In [ ]:
arrests_by_crime = (
    df_clean
    .dropna(subset=["num_arrests"])
    .groupby("crime_type")["num_arrests"]
    .agg(
        mean="mean",
        count="count"
    )
)

arrests_by_crime = (
    arrests_by_crime[
        arrests_by_crime["count"] >= 100
    ]
    .sort_values("mean")
)

fig, ax = plt.subplots(figsize=(10, 8))

bars = ax.barh(
    arrests_by_crime.index,
    arrests_by_crime["mean"]
)

ax.bar_label(
    bars,
    labels=[
        f"{value:.2f}"
        for value in arrests_by_crime["mean"]
    ],
    padding=4,
    fontsize=9
)

ax.set_title(
    "Average Number of Arrests by Crime Type",
    loc="left",
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Average arrests per incident")
ax.set_ylabel("")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

## Analytical EDA Findings

The analytical exploration provides several additional observations:

* Incident records are not evenly distributed across the 24-hour period, with some hours showing substantially higher recorded incident volumes than others.
* Suspect ages are concentrated around the middle of the 15–75 age range, while victim ages span a wider range from 10 to 90.
* Median property loss differs across crime types, although the magnitude of the differences should be interpreted within the context of the available observations.
* Average arrests vary across crime types, but the observed averages remain within a relatively narrow range.
* Filtering categories based on a minimum number of valid observations helps reduce misleading comparisons between well-represented and sparsely represented crime types.

These findings describe patterns observed in the dataset and should not be interpreted as causal relationships.


## 9. Relationship Analysis

This section examines relationships between multiple variables to identify differences in incident characteristics across crime types, districts, time periods, and severity levels.

Unlike the previous descriptive visualizations, these analyses compare variables simultaneously to provide more detailed insights into patterns within the dataset.

The analysis focuses on distributions and associations observed in the data and does not imply causal relationships.


### 9.1 Crime Type × Severity

This analysis examines the severity composition of the most frequently recorded crime types.

The top 15 crime types by incident volume are selected to improve readability. Percentages are calculated within each crime type using only records with non-missing severity values.

This allows the analysis to compare the severity composition of different crime types rather than simply comparing their total incident counts.


In [ ]:
severity_order = ["Low", "Medium", "High", "Critical"]

top_crimes = (
    df_clean["crime_type"]
    .value_counts()
    .head(15)
    .index
)

crime_severity = pd.crosstab(
    df_clean.loc[
        df_clean["crime_type"].isin(top_crimes),
        "crime_type"
    ],
    df_clean.loc[
        df_clean["crime_type"].isin(top_crimes),
        "severity"
    ],
    normalize="index"
) * 100

crime_severity = crime_severity.reindex(
    columns=severity_order,
    fill_value=0
)

crime_severity = crime_severity.sort_values(
    "Critical",
    ascending=True
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))

left = np.zeros(len(crime_severity))

for severity in severity_order:
    values = crime_severity[severity].values

    ax.barh(
        crime_severity.index,
        values,
        left=left,
        label=severity
    )

    left += values

ax.set_title(
    "Severity Composition by Crime Type",
    loc="left",
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Share of incidents with non-missing severity (%)")
ax.set_ylabel("")

ax.set_xlim(0, 100)

ax.legend(
    title="Severity",
    frameon=False,
    ncol=4,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.15)
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

### 9.2 District × Severity

This analysis compares the severity composition of incidents across districts.

A 100% stacked bar chart is used so that districts can be compared based on their severity proportions rather than their total incident volumes.

Missing severity values are excluded from the percentage calculation.


In [ ]:
district_severity = pd.crosstab(
    df_clean["district"],
    df_clean["severity"],
    normalize="index"
) * 100

district_severity = district_severity.reindex(
    columns=severity_order,
    fill_value=0
)

district_total = (
    df_clean.dropna(subset=["severity"])
    .groupby("district")
    .size()
    .reindex(district_severity.index)
)

district_severity["valid_n"] = district_total

district_severity = district_severity.sort_values(
    "valid_n",
    ascending=True
)

In [ ]:
plot_data = district_severity.drop(columns="valid_n")

fig, ax = plt.subplots(figsize=(11, 8))

left = np.zeros(len(plot_data))

for severity in severity_order:
    values = plot_data[severity].values

    ax.barh(
        plot_data.index,
        values,
        left=left,
        label=severity
    )

    left += values

ax.set_title(
    "Severity Composition by District",
    loc="left",
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Share of incidents with non-missing severity (%)")
ax.set_ylabel("")

ax.set_xlim(0, 100)

ax.legend(
    title="Severity",
    frameon=False,
    ncol=4,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.15)
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

### 9.3 Crime Type × Time of Day

This analysis examines how the distribution of incidents across the day varies by crime type.

The day is divided into four analytical periods based on incident hour:

* Night: 00:00–05:59
* Morning: 06:00–11:59
* Afternoon: 12:00–17:59
* Evening: 18:00–23:59

The periods are analytical groupings created for this analysis and are not original fields in the dataset.

The 10 most frequently recorded crime types are used to keep the visualization readable.


In [ ]:
time_period = pd.cut(
    df_clean["incident_datetime"].dt.hour,
    bins=[-1, 5, 11, 17, 23],
    labels=[
        "Night",
        "Morning",
        "Afternoon",
        "Evening"
    ]
)

top_10_crimes = (
    df_clean["crime_type"]
    .value_counts()
    .head(10)
    .index
)

crime_time = pd.crosstab(
    df_clean.loc[
        df_clean["crime_type"].isin(top_10_crimes),
        "crime_type"
    ],
    time_period.loc[
        df_clean["crime_type"].isin(top_10_crimes)
    ],
    normalize="index"
) * 100

time_order = [
    "Night",
    "Morning",
    "Afternoon",
    "Evening"
]

crime_time = crime_time.reindex(
    columns=time_order,
    fill_value=0
)

crime_time = crime_time.sort_values(
    "Night",
    ascending=True
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

left = np.zeros(len(crime_time))

for period in time_order:
    values = crime_time[period].values

    ax.barh(
        crime_time.index,
        values,
        left=left,
        label=period
    )

    left += values

ax.set_title(
    "Time-of-Day Composition by Crime Type",
    loc="left",
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Share of incidents (%)")
ax.set_ylabel("")

ax.set_xlim(0, 100)

ax.legend(
    title="Time period",
    frameon=False,
    ncol=4,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.15)
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.25
)

ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

### 9.4 Severity × Property Loss

This analysis examines the distribution of reported property loss across severity categories.

A boxplot is used to compare the median, spread, and potential outliers of property-loss values within each severity category.

Only records with both valid severity and property-loss values are included.


In [ ]:
# 1. Define order and an intuitive color palette (Green -> Red)
severity_order = ["Low", "Medium", "High", "Critical"]
severity_colors = {"Low": "#2ecc71", "Medium": "#f1c40f", "High": "#e67e22", "Critical": "#e74c3c"}

fig, ax = plt.subplots(figsize=(10, 6))

# 2. Base Boxplot (using seaborn removes the need for list comprehensions)
sns.boxplot(
    data=df_clean,
    x="severity",
    y="property_loss_usd",
    order=severity_order,
    palette=severity_colors,
    showfliers=False,         # Hide default fliers so they don't clash with stripplot
    width=0.5,
    boxprops=dict(alpha=0.7), # Add transparency to make overlay pop
    ax=ax
)

# 3. Add Stripplot overlay to visualize actual data density and distribution
sns.stripplot(
    data=df_clean,
    x="severity",
    y="property_loss_usd",
    order=severity_order,
    color="black",
    alpha=0.3,      # Make dots semi-transparent
    jitter=0.2,     # Spread points out horizontally
    size=4,
    ax=ax
)

# 4. Format Y-axis for financial readability ($1.5M instead of 1500000)
def currency_format(x, pos):
    if x >= 1e6:
        return f'${x*1e-6:.1f}M'
    elif x >= 1e3:
        return f'${x*1e-3:.0f}K'
    return f'${x:,.0f}'

ax.yaxis.set_major_formatter(FuncFormatter(currency_format))

# 5. Styling and labels
ax.set_title("Property Loss Distribution by Severity", loc="left", fontweight="bold", fontsize=14, pad=15)
ax.set_xlabel("Severity Level", fontweight="bold")
ax.set_ylabel("Property Loss (USD)", fontweight="bold")

# Clean up axes
sns.despine(ax=ax) # Replaces manually hiding top/right spines
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

## Relationship Analysis Findings

The relationship analysis provides several additional observations:

* The severity composition differs across crime types, indicating that incident volume alone does not fully describe the distribution of severity.
* Severity composition also varies across districts, although smaller districts should be interpreted cautiously because they contain fewer observations.
* Crime types exhibit different time-of-day compositions, suggesting that incident timing is not uniformly distributed across categories.
* Reported property loss has relatively similar central values across severity categories, indicating that severity classification and property loss do not show a large difference in their observed medians in this dataset.

These findings describe associations observed within the dataset and should not be interpreted as evidence of causality.


# Key Findings

The exploratory analysis of the cleaned crime dataset reveals several notable patterns.

### 1. Crime Distribution

The dataset contains 35 standardized crime types. DUI, drug offence, and domestic violence are among the most frequently recorded categories, while several other categories occur considerably less often.

### 2. Annual Incident Volume

The number of recorded incidents remains within a relatively narrow range between 2018 and 2024. The dataset contains 642 incidents in 2018 and 706 incidents in 2024, with moderate fluctuations between years.

### 3. Geographic Distribution

Incident volumes differ across districts. North and South contain the largest numbers of recorded incidents, while `mid` contains substantially fewer records than the other districts.

These differences represent the number of records in the dataset and should not be interpreted as crime rates because population or exposure data are not available.

### 4. Severity Composition

Medium and Critical incidents represent substantial portions of the dataset. The severity composition also differs across crime types and districts, indicating that incident frequency alone does not fully describe the distribution of severity.

### 5. Temporal Patterns

Incident records are not evenly distributed across the 24-hour day. Midnight has the highest recorded incident volume among individual hours, while several early-morning and daytime hours have lower volumes.

Crime types also show differences in their time-of-day composition.

### 6. Age Distribution

Recorded suspect ages are concentrated around the middle of the 15–75 valid range, with a median age of 46. Victim ages cover a wider range from 10 to 90, with a median age of 50.

### 7. Property Loss

Median reported property loss varies across crime types. However, the central values of property loss are relatively similar across severity categories, suggesting that severity and reported financial loss show limited separation in this dataset.

### 8. Arrests

The average number of arrests varies across crime types, but the differences remain relatively modest among categories with sufficient observations.

Overall, the analysis demonstrates that crime frequency, severity, location, timing, and financial impact provide different perspectives on the same dataset and should be considered together rather than interpreted in isolation.


In [ ]:
eda_summary = {
    "Total incidents": len(df_clean),
    "Crime types": df_clean["crime_type"].nunique(),
    "Districts": df_clean["district"].nunique(),
    "Cities": df_clean["city"].nunique(),
    "Highest-volume crime": df_clean["crime_type"].value_counts().idxmax(),
    "Highest-volume district": df_clean["district"].value_counts().idxmax(),
    "Highest-volume year": df_clean["incident_datetime"].dt.year.value_counts().idxmax(),
    "Most common severity": df_clean["severity"].value_counts().idxmax()
}

pd.Series(eda_summary)

# Conclusion & Limitations

## Conclusion

This project demonstrates a complete data-cleaning and exploratory-analysis workflow using Python and Pandas.

Starting from 5,250 records and 33 columns, the dataset was cleaned through duplicate removal, datetime standardization, numerical validation, geographic correction, categorical standardization, text normalization, and data-type conversion.

After cleaning, the final dataset contains **5,050 records and 33 columns**. All defined validation checks passed, including duplicate detection, numerical range checks, geographic coordinate validation, property-loss validation, and future-date validation.

The exploratory analysis then examined crime frequency, annual trends, geographic distribution, severity, incident timing, age distributions, property loss, arrests, and relationships between selected variables.

An important principle throughout the project was to avoid unsupported assumptions. When values could not be reliably recovered, they were retained as missing rather than being artificially reconstructed.

## Limitations

The dataset is synthetic and therefore should not be interpreted as a representation of real-world crime patterns.

The analysis also has several limitations:

* Population-level crime rates cannot be calculated because population or exposure data are not provided.
* Some identifier-to-attribute relationships remain inconsistent in the source data and were not forcibly reconstructed.
* Missing values remain in several columns because reliable imputation was not possible.
* Some category distinctions were intentionally preserved when merging them would require unsupported assumptions.
* The defined age ranges and other validation rules are analytical assumptions based on the observed dataset and should not be treated as universal domain standards.

Therefore, the findings describe patterns within this dataset rather than causal explanations or real-world crime statistics.


# Export Clean Dataset

The validated dataset is exported as a CSV file for further analysis or reuse.

The exported file contains the cleaned records and retains the final data types and missing-value representation required for downstream analysis.


In [ ]:
output_file = "cleaned_crime_dataset.csv"

df_clean.to_csv(
    output_file,
    index=False
)

print(f"Dataset exported to: {output_file}")

In [ ]:
df_exported = pd.read_csv(output_file)

print("Exported shape:", df_exported.shape)
print("File exists:", output_file)

In [ ]:
assert df_exported.shape == df_clean.shape
assert df_exported.columns.tolist() == df_clean.columns.tolist()

print("Export verification passed.")